In [47]:
!pip install -r requirements.txt

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Configuración inicial
base_url = "https://www.mercadolibre.com.ec/ofertas"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
}

lista_final = []

# Definimos cuántas páginas queremos recorrer 
paginas_a_scrapear = 21

print(f"Iniciando extracción de {paginas_a_scrapear} páginas de ofertas...")

for i in range(paginas_a_scrapear):
    # Mercado Libre usa el parámetro _Desde_ seguido de múltiplos de 48 para paginar

    offset = (i * 48) + 1
    url = f"{base_url}?page={i+1}" if i > 0 else base_url
    
    print(f"Extrayendo datos de la página {i+1}...")
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        productos = soup.select('div.poly-card__content') or soup.find_all('div', class_='promotion-item__container')

        for item in productos:
            # 1. Título
            titulo_tag = item.find(['h2', 'h3', 'p'], class_=lambda x: x and 'title' in x)
            titulo = titulo_tag.get_text(strip=True) if titulo_tag else "Sin nombre"

            # 2. Precio con decimales (Tu observación clave)
            promo_container = item.select_one('span.andes-money-amount--promotion, div.poly-price__current')
            
            if promo_container:
                f_tag = promo_container.select_one('span.andes-money-amount__fraction')
                c_tag = promo_container.select_one('span.andes-money-amount__cents')
                
                entero = f_tag.get_text(strip=True).replace('.', '') if f_tag else "0"
                centavos = c_tag.get_text(strip=True) if c_tag else "00"
                precio_final = float(f"{entero}.{centavos}")
            else:
                precio_final = 0.0

            # 3. Enlace
            link_tag = item.find('a', href=True)
            enlace = link_tag['href'] if link_tag else "N/A"

            lista_final.append({
                "Página": i + 1,
                "Producto": titulo,
                "Precio_Oferta": precio_final,
                "Link": enlace
            })
        
        # Pausa breve para no saturar el servidor y evitar bloqueos (buena práctica)
        time.sleep(1)

    except Exception as e:
        print(f"Error en página {i+1}: {e}")
        break

# --- GUARDADO ---
df = pd.DataFrame(lista_final)
df.to_csv('scrapping.csv', index=False, encoding='utf-8-sig')

print(f"¡Proceso terminado! Total de productos extraídos: {len(df)}")
print(f"Los datos se guardaron en 'scrapping.csv'")

Iniciando extracción de 21 páginas de ofertas...
Extrayendo datos de la página 1...
Extrayendo datos de la página 2...
Extrayendo datos de la página 3...
Extrayendo datos de la página 4...
Extrayendo datos de la página 5...
Extrayendo datos de la página 6...
Extrayendo datos de la página 7...
Extrayendo datos de la página 8...
Extrayendo datos de la página 9...
Extrayendo datos de la página 10...
Extrayendo datos de la página 11...
Extrayendo datos de la página 12...
Extrayendo datos de la página 13...
Extrayendo datos de la página 14...
Extrayendo datos de la página 15...
Extrayendo datos de la página 16...
Extrayendo datos de la página 17...
Extrayendo datos de la página 18...
Extrayendo datos de la página 19...
Extrayendo datos de la página 20...
Extrayendo datos de la página 21...
¡Proceso terminado! Total de productos extraídos: 1008
Los datos se guardaron en 'todas_las_ofertas_ml.csv'
